# Post-hoc GNNExplainer on Vanilla 4-layer GINE (Custom BA2Motif)

This notebook trains a vanilla 4-layer **GINE** classifier on your custom BA2Motif dataset (with ground-truth `node_mask`), then fits a **post-hoc GNNExplainer** and evaluates **Jaccard@|GT|** and **Node AUROC**.

Dataset file used: `/home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique/shaique_updates/data/Custom_BA2Motif/custom_ba2motif_dataset.pt`

In [11]:
import os
import sys
from pathlib import Path
import random
import numpy as np
import torch

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import degree

# -------------------------
# Repro / device
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

# -------------------------
# Ensure repo root on sys.path (so `import bcosgnn` works)
# -------------------------
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
print('Repo root added:', repo_root)

DEVICE = cpu
Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [12]:
# -------------------------
# Load custom BA2Motif dataset list
# -------------------------
RAW_DATA_PATH = "/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/shaique_updates/data/Custom_BA2Motif/custom_ba2motif_dataset.pt"
assert os.path.exists(RAW_DATA_PATH), f"Missing file: {RAW_DATA_PATH}"
print('Loading:', RAW_DATA_PATH)

# Enforce loading to CPU
dataset_list = torch.load(RAW_DATA_PATH, map_location='cpu', weights_only=False)
print('Graphs loaded:', len(dataset_list))
assert isinstance(dataset_list, (list, tuple)) and len(dataset_list) > 0

# If labels are missing, assign first half=0, second half=1 (matches prior BA2Motif scripts).
for i, data in enumerate(dataset_list):
    # Ensure all data is definitely on CPU
    data = data.cpu()
    dataset_list[i] = data

    if not hasattr(data, 'y') or data.y is None or data.y.numel() == 0:
        data.y = torch.tensor([0 if i < (len(dataset_list) // 2) else 1], dtype=torch.long)
    else:
        # Re-assigning .to(torch.long) typically preserves device, but data is CPU now
        data.y = data.y.view(-1).to(torch.long)
        if data.y.numel() == 1:
            data.y = data.y.view(1)
        else:
            # graph-level label: keep first entry if provided oddly
            data.y = data.y[:1]

# Build node features from (clamped) degree one-hot so vanilla GIN has non-trivial inputs.
def add_degree_onehot_x(data: Data, max_degree: int = 4) -> Data:
    row, col = data.edge_index
    deg = degree(col, data.num_nodes, dtype=torch.long).clamp(max=max_degree)
    data.x = torch.nn.functional.one_hot(deg, num_classes=max_degree + 1).to(torch.float)
    return data

for i in range(len(dataset_list)):
    # Only add node features, no edge attributes for GIN
    dataset_list[i] = add_degree_onehot_x(dataset_list[i])
    # Remove edge_attr if it exists to ensure GNNExplainer doesn't get confused
    if hasattr(dataset_list[i], 'edge_attr'):
        del dataset_list[i].edge_attr

# Basic dataset stats
ys = torch.cat([d.y for d in dataset_list]).view(-1)
num_classes = int(torch.unique(ys).numel())
in_dim = int(dataset_list[0].x.size(-1))

print('num_classes =', num_classes)
print('node feature dim =', in_dim)
print('edge feature dim = N/A (GIN)')

Loading: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/shaique_updates/data/Custom_BA2Motif/custom_ba2motif_dataset.pt
Graphs loaded: 1000
num_classes = 2
node feature dim = 5
edge feature dim = N/A (GIN)
Graphs loaded: 1000
num_classes = 2
node feature dim = 5
edge feature dim = N/A (GIN)


In [13]:
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU, BatchNorm1d
from torch_geometric.nn import GINConv, global_add_pool

class VanillaGIN4(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, drop_ratio: float = 0.5):
        super().__init__()
        self.node_proj = Linear(in_dim, hidden_dim)

        def mlp():
            return Sequential(
                Linear(hidden_dim, 2 * hidden_dim),
                BatchNorm1d(2 * hidden_dim),
                ReLU(),
                Linear(2 * hidden_dim, hidden_dim),
            )

        self.conv1 = GINConv(nn=mlp(), train_eps=True)
        self.bn1 = BatchNorm1d(hidden_dim)
        self.conv2 = GINConv(nn=mlp(), train_eps=True)
        self.bn2 = BatchNorm1d(hidden_dim)
        self.conv3 = GINConv(nn=mlp(), train_eps=True)
        self.bn3 = BatchNorm1d(hidden_dim)
        self.conv4 = GINConv(nn=mlp(), train_eps=True)
        self.bn4 = BatchNorm1d(hidden_dim)

        self.drop_ratio = drop_ratio
        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch=None, edge_attr=None):
        # Arguments ordered: batch first (after edge_index), then edge_attr
        # This supports positional calls: model(x, ei, batch)
        # And keyword calls from Explainer: model(x, ei, ..., edge_attr=ea)
        
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        h = self.node_proj(x.float())

        h = F.relu(self.bn1(self.conv1(h, edge_index)))
        h = F.relu(self.bn2(self.conv2(h, edge_index)))
        h = F.relu(self.bn3(self.conv3(h, edge_index)))
        h = F.relu(self.bn4(self.conv4(h, edge_index)))

        hg = global_add_pool(h, batch)
        hg = F.dropout(hg, p=self.drop_ratio, training=self.training)
        return self.classifier(hg)

# Model instantiation moved to Experiment Loop
# model = VanillaGIN4(in_dim=in_dim, hidden_dim=64, num_classes=num_classes, drop_ratio=0.5).to(DEVICE)


In [14]:
from copy import deepcopy
from tqdm import tqdm
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(DEVICE)
        if batch.num_nodes <= 1:
            continue

        # GIN does not use edge_attr
        logits = model(batch.x, batch.edge_index, batch.batch)
        y = batch.y.view(-1).to(torch.long)

        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * int(batch.num_graphs)
        pred = logits.argmax(dim=-1)
        total_correct += int((pred == y).sum().item())
        total_graphs += int(batch.num_graphs)

    return (total_loss / max(total_graphs, 1)), (total_correct / max(total_graphs, 1))

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(DEVICE)
        if batch.num_nodes <= 1:
            continue

        logits = model(batch.x, batch.edge_index, batch.batch)
        y = batch.y.view(-1).to(torch.long)
        loss = criterion(logits, y)

        total_loss += float(loss.item()) * int(batch.num_graphs)
        pred = logits.argmax(dim=-1)
        total_correct += int((pred == y).sum().item())
        total_graphs += int(batch.num_graphs)

    return (total_loss / max(total_graphs, 1)), (total_correct / max(total_graphs, 1))

def get_gnn_explainer(model, epochs: int = 200, lr: float = 0.01):
    # We remove edge_mask_type='object' because our GIN model doesn't use edge attributes.
    return Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=epochs, lr=lr),
        explanation_type='model',
        node_mask_type='object',
        edge_mask_type=None,
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )


In [15]:
import time
import torch
import numpy as np

class CUDATimer:
    """Accurately measures execution time on GPU."""
    def __init__(self, name="Process"):
        self.name = name
        self.times = [] # Stores time in milliseconds
    
    def __enter__(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.end = time.perf_counter()
        self.times.append((self.end - self.start) * 1000) # Convert to ms

    def get_stats(self):
        return np.mean(self.times), np.std(self.times)

In [16]:
# -------------------------
# Timing helpers for GNNExplainer
# -------------------------
import time
import bcosgnn.evaluation

def _sync_if_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def time_gnnexplainer_per_graph(gnn_explainer, model, dataset, warmup: int = 2, max_graphs: int | None = None):
    """Returns per-graph timing stats (ms) for GNNExplainer on a dataset list."""
    model.eval()
    times_ms = []
    graphs = dataset[:max_graphs] if max_graphs is not None else dataset

    # Warmup (not timed)
    for i, data in enumerate(graphs[:warmup]):
        data = data.to(DEVICE)
        _ = bcosgnn.evaluation.get_gnnexplainer_scores(gnn_explainer, model, data)

    for data in graphs:
        data = data.to(DEVICE)
        _sync_if_cuda()
        start = time.perf_counter()
        _ = bcosgnn.evaluation.get_gnnexplainer_scores(gnn_explainer, model, data)
        _sync_if_cuda()
        end = time.perf_counter()
        times_ms.append((end - start) * 1000)

    times_ms = np.asarray(times_ms, dtype=float)
    mean_ms = float(times_ms.mean()) if times_ms.size else float('nan')
    std_ms = float(times_ms.std()) if times_ms.size else float('nan')
    median_ms = float(np.median(times_ms)) if times_ms.size else float('nan')
    p90_ms = float(np.percentile(times_ms, 90)) if times_ms.size else float('nan')
    graphs_per_s = float(1000.0 / mean_ms) if mean_ms > 0 else float('nan')
    total_s = float(times_ms.sum() / 1000.0) if times_ms.size else float('nan')

    return {
        "mean_ms": mean_ms,
        "std_ms": std_ms,
        "median_ms": median_ms,
        "p90_ms": p90_ms,
        "graphs_per_s": graphs_per_s,
        "total_s": total_s,
        "n_graphs": int(times_ms.size),
    }

In [17]:
# -------------------------
# ICML Style Reporting: Multi-Seed Experiment
# -------------------------
from bcosgnn.evaluation import evaluate_gnnexplainer_jaccard, evaluate_gnnexplainer_auroc
import time

SEEDS = [0, 1, 2, 3, 4]
results_jaccard = []
results_auroc = []
results_test_acc = []
results_gnn_time_mean_ms = []
results_gnn_time_median_ms = []
results_gnn_time_p90_ms = []
results_gnn_time_graphs_per_s = []
results_gnn_train_time_s = []
results_gnn_explain_total_s = []
results_gnn_end_to_end_s = []

for seed in SEEDS:
    print(f"\n{'='*20} SEED {seed} {'='*20}")
    
    # 1. Set Seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 2. Split Data (Random split per seed)
    indices = np.arange(len(dataset_list))
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    
    n = len(indices)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]
    
    train_dataset = [dataset_list[i] for i in train_idx]
    val_dataset = [dataset_list[i] for i in val_idx]
    test_dataset = [dataset_list[i] for i in test_idx]
    
    BATCH_SIZE = 128
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 3. Initialize Model
    model = VanillaGIN4(in_dim=in_dim, hidden_dim=64, num_classes=num_classes, drop_ratio=0.5).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = torch.nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
    )
    
    # 4. Train
    EPOCHS = 100
    EARLY_STOP_PATIENCE = 25
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    
    _sync_if_cuda()
    train_start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = eval_one_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epoch % 50 == 0:
             print(f"  Epoch {epoch:03d}: val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break
            
    _sync_if_cuda()
    train_end = time.perf_counter()
    train_time_s = train_end - train_start
    results_gnn_train_time_s.append(train_time_s)
    print(f"  Training time: {train_time_s:.2f}s")
            
    if best_state is not None:
        model.load_state_dict(best_state)
    
    test_loss, test_acc = eval_one_epoch(model, test_loader, criterion)
    results_test_acc.append(test_acc)
    print(f"  Best Val Loss: {best_val_loss:.4f} | Test Acc: {test_acc:.4f}")
    
    # 5. Explainer Evaluation
    print("  Evaluating Explainer...")
    gnn_explainer = get_gnn_explainer(model, epochs=200, lr=0.01)
    
    jacc = evaluate_gnnexplainer_jaccard(gnn_explainer, model, test_dataset)
    auc = evaluate_gnnexplainer_auroc(gnn_explainer, model, test_dataset)
    
    results_jaccard.append(jacc)
    results_auroc.append(auc)
    print(f"  Seed {seed} Result -> Jaccard: {jacc:.4f}, AUROC: {auc:.4f}")
    
    # 6. Timing (per-graph, end-to-end GNNExplainer)
    timing = time_gnnexplainer_per_graph(
        gnn_explainer, model, test_dataset, warmup=2, max_graphs=None
    )
    results_gnn_time_mean_ms.append(timing["mean_ms"])
    results_gnn_time_median_ms.append(timing["median_ms"])
    results_gnn_time_p90_ms.append(timing["p90_ms"])
    results_gnn_time_graphs_per_s.append(timing["graphs_per_s"])
    results_gnn_explain_total_s.append(timing["total_s"])
    end_to_end_s = train_time_s + timing["total_s"]
    results_gnn_end_to_end_s.append(end_to_end_s)
    print(
        f"  Timing (ms/graph): mean={timing['mean_ms']:.2f}, "
        f"median={timing['median_ms']:.2f}, p90={timing['p90_ms']:.2f} "
        f"| throughput={timing['graphs_per_s']:.2f} graphs/s"
    )
    print(f"  Explain total time: {timing['total_s']:.2f}s | End-to-end: {end_to_end_s:.2f}s")

# 7. Report Stats
print("\n" + "#"*40)
print("FINAL RESULTS (Mean \u00B1 Std)")
print("#"*40)
print(f"Test Acc:      {np.mean(results_test_acc):.4f} \u00B1 {np.std(results_test_acc):.4f}")
print(f"Jaccard@|GT|:  {np.mean(results_jaccard):.4f} \u00B1 {np.std(results_jaccard):.4f}")
print(f"Node AUROC:    {np.mean(results_auroc):.4f} \u00B1 {np.std(results_auroc):.4f}")
print(f"GNN train time (s): {np.mean(results_gnn_train_time_s):.2f} \u00B1 {np.std(results_gnn_train_time_s):.2f}")
print(f"GNN explain total time (s): {np.mean(results_gnn_explain_total_s):.2f} \u00B1 {np.std(results_gnn_explain_total_s):.2f}")
print(f"GNN end-to-end time (s): {np.mean(results_gnn_end_to_end_s):.2f} \u00B1 {np.std(results_gnn_end_to_end_s):.2f}")
print(f"GNNExplainer mean ms/graph: {np.mean(results_gnn_time_mean_ms):.2f} \u00B1 {np.std(results_gnn_time_mean_ms):.2f}")
print(f"GNNExplainer median ms/graph: {np.mean(results_gnn_time_median_ms):.2f} \u00B1 {np.std(results_gnn_time_median_ms):.2f}")
print(f"GNNExplainer p90 ms/graph: {np.mean(results_gnn_time_p90_ms):.2f} \u00B1 {np.std(results_gnn_time_p90_ms):.2f}")
print(f"GNNExplainer throughput (graphs/s): {np.mean(results_gnn_time_graphs_per_s):.2f} \u00B1 {np.std(results_gnn_time_graphs_per_s):.2f}")



==================== SEED 0 ====================
  Early stopping at epoch 36
  Training time: 7.15s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...
  Early stopping at epoch 36
  Training time: 7.15s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...


Evaluating Jaccard (gnnexplainer):   0%|          | 0/100 [00:00<?, ?it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   1%|          | 1/100 [00:00<00:27,  3.60it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   2%|▏   

  Seed 0 Result -> Jaccard: 0.7468, AUROC: 0.9334
  Timing (ms/graph): mean=274.42, median=275.30, p90=278.47 | throughput=3.64 graphs/s
  Explain total time: 27.44s | End-to-end: 34.59s

==================== SEED 1 ====================
  Timing (ms/graph): mean=274.42, median=275.30, p90=278.47 | throughput=3.64 graphs/s
  Explain total time: 27.44s | End-to-end: 34.59s

==================== SEED 1 ====================
  Early stopping at epoch 40
  Training time: 4.90s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...
  Early stopping at epoch 40
  Training time: 4.90s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...


Evaluating AUROC (gnnexplainer):   1%|          | 1/100 [00:00<00:27,  3.62it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   2%|▏         | 2/100 [00:00<00:26,  3.65it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   3

  Seed 1 Result -> Jaccard: 0.4604, AUROC: 0.7903
  Timing (ms/graph): mean=272.71, median=273.30, p90=276.22 | throughput=3.67 graphs/s
  Explain total time: 27.27s | End-to-end: 32.17s

==================== SEED 2 ====================
  Timing (ms/graph): mean=272.71, median=273.30, p90=276.22 | throughput=3.67 graphs/s
  Explain total time: 27.27s | End-to-end: 32.17s

==================== SEED 2 ====================
  Early stopping at epoch 31
  Training time: 3.81s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...
  Early stopping at epoch 31
  Training time: 3.81s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...


...

  Seed 2 Result -> Jaccard: 0.6365, AUROC: 0.8258
  Timing (ms/graph): mean=274.90, median=276.23, p90=278.34 | throughput=3.64 graphs/s
  Explain total time: 27.49s | End-to-end: 31.30s

==================== SEED 3 ====================
  Timing (ms/graph): mean=274.90, median=276.23, p90=278.34 | throughput=3.64 graphs/s
  Explain total time: 27.49s | End-to-end: 31.30s

==================== SEED 3 ====================
  Early stopping at epoch 31
  Training time: 3.81s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...
  Early stopping at epoch 31
  Training time: 3.81s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...


Evaluating AUROC (gnnexplainer):   1%|          | 1/100 [00:00<00:27,  3.58it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   2%|▏         | 2/100 [00:00<00:27,  3.61it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   3

  Seed 3 Result -> Jaccard: 0.3710, AUROC: 0.6695
  Timing (ms/graph): mean=270.09, median=270.93, p90=274.17 | throughput=3.70 graphs/s
  Explain total time: 27.01s | End-to-end: 30.82s

==================== SEED 4 ====================
  Timing (ms/graph): mean=270.09, median=270.93, p90=274.17 | throughput=3.70 graphs/s
  Explain total time: 27.01s | End-to-end: 30.82s

==================== SEED 4 ====================
  Early stopping at epoch 32
  Training time: 4.02s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...
  Early stopping at epoch 32
  Training time: 4.02s
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...


Evaluating AUROC (gnnexplainer):   1%|          | 1/100 [00:00<00:28,  3.49it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   2%|▏         | 2/100 [00:00<00:27,  3.57it/s]/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/bcosgnn/evaluation.py:77: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating AUROC (gnnexplainer):   3

  Seed 4 Result -> Jaccard: 0.6686, AUROC: 0.9010
  Timing (ms/graph): mean=287.21, median=282.25, p90=297.46 | throughput=3.48 graphs/s
  Explain total time: 28.72s | End-to-end: 32.74s

########################################
FINAL RESULTS (Mean ± Std)
########################################
Test Acc:      1.0000 ± 0.0000
Jaccard@|GT|:  0.5766 ± 0.1391
Node AUROC:    0.8240 ± 0.0926
GNN train time (s): 4.74 ± 1.27
GNN explain total time (s): 27.59 ± 0.59
GNN end-to-end time (s): 32.32 ± 1.32
GNNExplainer mean ms/graph: 275.87 ± 5.92
GNNExplainer median ms/graph: 275.60 ± 3.79
GNNExplainer p90 ms/graph: 280.93 ± 8.41
GNNExplainer throughput (graphs/s): 3.63 ± 0.08
  Timing (ms/graph): mean=287.21, median=282.25, p90=297.46 | throughput=3.48 graphs/s
  Explain total time: 28.72s | End-to-end: 32.74s

########################################
FINAL RESULTS (Mean ± Std)
########################################
Test Acc:      1.0000 ± 0.0000
Jaccard@|GT|:  0.5766 ± 0.1391
Node AUROC:    

In [39]:
# -------------------------
# Configure post-hoc GNNExplainer
# -------------------------
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig

def get_gnn_explainer(model, epochs: int = 200, lr: float = 0.01):
    # We train a multiclass classifier with `num_classes` logits.
    # We remove edge_mask_type='object' because our GIN model doesn't use edge attributes.
    return Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=epochs, lr=lr),
        explanation_type='model',
        node_mask_type='object',
        edge_mask_type=None,  # No edge attribute masking
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

gnn_explainer = get_gnn_explainer(model, epochs=200, lr=0.01)
print(gnn_explainer)

In [ ]:
# -------------------------
# Evaluate explanations (Node AUROC + Jaccard@|GT|)
# -------------------------
from bcosgnn.evaluation import evaluate_gnnexplainer_jaccard, evaluate_gnnexplainer_auroc

# Important: run explanations per-graph to avoid GT/pred mismatches.
test_dataset_per_graph = test_dataset  # list[Data]
jaccard = evaluate_gnnexplainer_jaccard(gnn_explainer, model, test_dataset_per_graph)
auroc = evaluate_gnnexplainer_auroc(gnn_explainer, model, test_dataset_per_graph)

print("\n--- GNNExplainer on Vanilla 4-layer GIN ---")
print(f"Jaccard@|GT|: {jaccard:.4f}")
print(f"Node AUROC:   {auroc:.4f}")

Evaluating AUROC (gnnexplainer):   1%|          | 1/100 [00:01<02:26,  1.48s/it]/home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique/shaique_updates/codes/venv/lib/python3.12/site-packages/torch_geometric/explain/explainer.py:193: UserWarning: The 'target' should not be provided for the explanation type 'model'
  warnings.warn(
/home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique/shaique_updates/codes/venv/lib/python3.12/site-packages/torch_geometric/explain/explainer.py:193: UserWarning: The 'target' should not be provided for the explanation type 'model'
  warnings.warn(
Evaluating AUROC (gnnexplainer):   2%|▏         | 2/100 [00:02<02:23,  1.46s/it]/home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique/shaique_updates/codes/venv/lib/python3.12/site-packages/torch_geometric/explain/explainer.py:193: UserWarning: The 'target' should not be provided for the explanation type 'model'
  warnings.warn(
/home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique/shaique_updates/codes/venv/lib/python


--- GNNExplainer on Vanilla 4-layer GINE ---
Jaccard@|GT|: 0.5568
Node AUROC:   0.8352


In [31]:
# DEBUG: Check single graph explanation scores
import bcosgnn.evaluation
print("Checking single graph explanation...")
data = test_dataset[0].to(DEVICE)

# Verify valid Ground Truth mask exists
if hasattr(data, 'node_mask'):
    print(f"Ground Truth node_mask sum: {data.node_mask.sum().item()} / {data.num_nodes}")
else:
    print("WARNING: Data check - No node_mask found in test data!")

try:
    scores, target = bcosgnn.evaluation.get_gnnexplainer_scores(
        gnn_explainer, model, data
    )
    print("Scores shape:", scores.shape)
    print("Scores stats: min={:.4f}, max={:.4f}, mean={:.4f}".format(scores.min(), scores.max(), scores.mean()))
    print("Target class:", target)
except Exception as e:
    print("Error:", e)


Checking single graph explanation...
Ground Truth node_mask sum: 5 / 30
Error: VanillaGIN4.forward() got an unexpected keyword argument 'edge_attr'
